# Build the curated analytics table

Reads the raw Parquet mirror written by `Export_to_parquet.ipynb`, flattens the
single annual statement out of `data[0]`, joins the register attributes, derives
operating margin, and writes one row per company as a single Parquet file for
PowerBI.

**Scope.** Only the 444,644 companies with a filed statement
(`fetch_status = "success"`). Companies without a filing are out of scope and
are not represented as rows. All legal forms are kept; `organisasjonsform_kode`
is a column, so an AS-only view is a downstream filter rather than a decision
baked into the table.

**Operating margin.** `(driftsresultat / sumDriftsinntekter) * 100`, in percent,
stored unrounded. Null whenever the ratio is undefined or misleading, with the
reason recorded in `operating_margin_status`. Precedence, first match wins:

| status | condition |
|---|---|
| `revenue_missing` | `sumDriftsinntekter` absent |
| `revenue_zero` | revenue exactly 0 |
| `revenue_negative` | revenue < 0 |
| `income_missing` | revenue > 0 but `driftsresultat` absent |
| `computed` | everything else; margin is populated |

The zero test is an exact floating-point comparison. That is safe here because
the API reports whole NOK amounts, so an intended zero is stored as exactly
`0.0` rather than as a rounding residue.

**Numerator and denominator are kept as columns.** A group-level operating
margin is `SUM(operating_income) / SUM(revenue)` — the pooled margin — which is
not the mean of the per-company margins. Given the skew, the two differ
substantially. Keeping the components lets PowerBI compute the pooled figure
with `DIVIDE(SUM(...), SUM(...))` while the row-level column serves
distributional analysis.

**No winsorised column.** The margin is stored unbounded. A company with 5,000
NOK revenue and a 2 MNOK operating loss yields -40,000% legitimately; that is
signal about the register, not an error. Outlier handling belongs in the
analysis notebook, where it can be varied.

**Comparability is flagged, not filtered.** `avviklingsregnskap` and
`is_full_year` mark liquidation accounts and non-annual periods. Both stay in
the table.

**Filesystem note.** `data/` is a bind mount onto a Dropbox-synced Windows
folder. Spark's directory-based output is therefore staged on container-local
disk under `/tmp`, and only the finished single file is copied onto the mount.
Writing a Spark output directory directly into `data/` fails on cleanup with
`PermissionError` on `rmdir`, because `shutil.rmtree` uses file-descriptor-
relative calls that the bind mount does not reliably support, and because
Dropbox may hold a handle on the directory. Pausing Dropbox sync before a run
is still advisable.

In [1]:
import glob
import json
import os
import shutil
import statistics
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

DATA_DIR = "/home/jovyan/data"
PARQUET_DIR = os.path.join(DATA_DIR, "parquet")

# Single file rather than a part-file directory, so PowerBI's Parquet connector
# takes a plain path and needs no Folder/Combine step.
ANALYTICS_FILE = os.path.join(PARQUET_DIR, "analytics_company_financials.parquet")

# All Spark directory output goes to container-local disk. data/ is a bind mount
# onto a Dropbox-synced Windows folder where removing a directory fails
# intermittently even though writing into it succeeds, so only finished single
# files are placed on the mount.
STAGING_DIR = "/tmp/group13_analytics_staging"
SCRATCH_DIR = "/tmp/group13_scalability"

spark = SparkSession.builder.appName("group13_build_analytics").getOrCreate()

# Captured now rather than in the summary cell: the scalability experiment
# stops this session, after which spark.version is no longer reachable.
SPARK_VERSION = spark.version

print("Spark        ", SPARK_VERSION)
print("master       ", spark.sparkContext.master)
print("driver heap  %.1f GB" % (spark._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3))
print("parallelism  ", spark.sparkContext.defaultParallelism)
print("host cores   ", os.cpu_count())

Spark         4.2.0
master        local[4]
driver heap  8.0 GB
parallelism   4
host cores    12


## The build

One function, so the scalability experiment further down runs exactly the code
that produces the deliverable rather than a simplified stand-in.

`data` is typed as an array because that is what the API returns, but the
profiling pass confirmed exactly one element in all 444,644 populated records,
so element 0 is taken directly and the join stays one-to-one. No `explode`.

In [2]:
# Paths into the nested statement. Named once so a typo cannot silently produce
# a column of nulls in one place and not another.
_RES = "d.resultatregnskapResultat"
_DR = _RES + ".driftsresultat"


# Returns the curated DataFrame: one row per company with a filed statement.
# Nothing is cached and no action is triggered, so the caller decides when and
# how the work is materialised. Taking a session argument rather than closing
# over a global is what lets the scalability experiment re-run this exact code
# under a different master.
def build(session):
    companies = session.read.parquet(os.path.join(PARQUET_DIR, "companies")).select(
        "organisasjonsnummer",
        F.col("navn"),
        F.col("organisasjonsform.kode").alias("organisasjonsform_kode"),
        F.col("naeringskode1.kode").alias("naeringskode1_kode"),
        F.col("naeringskode1.beskrivelse").alias("naeringskode1_beskrivelse"),
        F.col("forretningsadresse.kommunenummer").alias("kommunenummer"),
        F.col("forretningsadresse.kommune").alias("kommune"),
        F.col("antallAnsatte").alias("antall_ansatte"),
        F.col("stiftelsesdato").alias("stiftelsesdato"),
        F.col("registreringsdatoEnhetsregisteret").alias("registrert_dato"),
        F.col("konkurs"),
        F.col("underAvvikling").alias("under_avvikling"),
    )

    financial = (
        session.read.parquet(os.path.join(PARQUET_DIR, "financial_data"))
        .filter("fetch_status = 'success'")
        .select("organisasjonsnummer", F.col("data")[0].alias("d"))
    )

    revenue = F.col(_DR + ".driftsinntekter.sumDriftsinntekter")
    income = F.col(_DR + ".driftsresultat")

    # A margin is only meaningful with a strictly positive denominator and a
    # present numerator. Anything else is null, with the cause recorded.
    computable = revenue.isNotNull() & (revenue > 0) & income.isNotNull()

    status = (
        F.when(revenue.isNull(), "revenue_missing")
        .when(revenue == 0, "revenue_zero")
        .when(revenue < 0, "revenue_negative")
        .when(income.isNull(), "income_missing")
        .otherwise("computed")
    )

    margin = F.when(computable, (income / revenue) * 100).otherwise(
        F.lit(None).cast("double"))

    period_from = F.to_date(F.col("d.regnskapsperiode.fraDato"))
    period_to = F.to_date(F.col("d.regnskapsperiode.tilDato"))
    # Inclusive of both endpoints: 2025-01-01 to 2025-12-31 is 365 days.
    period_days = F.datediff(period_to, period_from) + 1

    return (
        financial.join(companies, "organisasjonsnummer")
        .select(
            # Register identity and attributes
            "organisasjonsnummer", "navn", "organisasjonsform_kode",
            "naeringskode1_kode", "naeringskode1_beskrivelse",
            "kommunenummer", "kommune", "antall_ansatte",
            "stiftelsesdato", "registrert_dato", "konkurs", "under_avvikling",

            # Filing metadata, including the two comparability flags
            F.col("d.regnskapstype").alias("regnskapstype"),
            F.col("d.oppstillingsplan").alias("oppstillingsplan"),
            F.col("d.valuta").alias("valuta"),
            F.col("d.virksomhet.morselskap").alias("morselskap"),
            F.col("d.regnkapsprinsipper.smaaForetak").alias("smaa_foretak"),
            F.col("d.avviklingsregnskap").alias("avviklingsregnskap"),
            period_from.alias("period_from"),
            period_to.alias("period_to"),
            period_days.alias("period_days"),
            (period_days.isin(365, 366)).alias("is_full_year"),

            # Income statement and balance sheet
            revenue.alias("revenue"),
            income.alias("operating_income"),
            F.col(_DR + ".driftskostnad.sumDriftskostnad").alias("operating_costs"),
            F.col("d.egenkapitalGjeld.egenkapital.sumEgenkapital").alias("sum_egenkapital"),
            F.col("d.egenkapitalGjeld.gjeldOversikt.sumGjeld").alias("sum_gjeld"),
            F.col("d.eiendeler.sumEiendeler").alias("sum_eiendeler"),

            # Derived
            margin.alias("operating_margin_pct"),
            status.alias("operating_margin_status"),
        )
    )

## Input profile

Run before the write. These are the frequencies that justify the null rules, and
they belong in the report — the size of each undefined-margin class is a
data-quality result, not a footnote. Counted over all 444,644 filings, not a
sample.

In [3]:
curated = build(spark).cache()

# Both sides are measured rather than compared against a stored constant. The
# property under test is that the join is one-to-one: the curated table must
# have exactly one row per successful filing, no more (duplicate register rows)
# and no fewer (filings whose organisasjonsnummer is absent from companies).
n_source = (spark.read.parquet(os.path.join(PARQUET_DIR, "financial_data"))
            .filter("fetch_status = 'success'").count())
n_rows = curated.count()
n_distinct = curated.select("organisasjonsnummer").distinct().count()

print("Successful filings in mirror : %d" % n_source)
print("Rows in curated table        : %d  %s"
      % (n_rows, "OK" if n_rows == n_source else "MISMATCH"))
print("Distinct organisasjonsnummer : %d  %s"
      % (n_distinct, "OK" if n_distinct == n_rows else "DUPLICATES"))
assert n_rows == n_source, "join changed cardinality"
assert n_distinct == n_rows, "duplicate organisasjonsnummer in the register mirror"

print("\nOperating margin status over all %d filings:" % n_rows)
status_rows = (curated.groupBy("operating_margin_status").count()
               .orderBy(F.desc("count")).collect())
for r in status_rows:
    print("  %-18s %8d  %6.2f%%"
          % (r["operating_margin_status"], r["count"], 100.0 * r["count"] / n_rows))

status_counts = {r["operating_margin_status"]: r["count"] for r in status_rows}

print("\nComparability flags:")
flag_counts = {}
for col in ["avviklingsregnskap", "is_full_year", "morselskap", "smaa_foretak"]:
    flag_counts[col] = curated.filter(F.col(col)).count()
    print("  %-20s true in %8d  %6.2f%%"
          % (col, flag_counts[col], 100.0 * flag_counts[col] / n_rows))

print("\nCurrency (a ratio is scale-invariant, so this does not distort the margin):")
valuta_counts = {r["valuta"]: r["count"]
                 for r in curated.groupBy("valuta").count().orderBy(F.desc("count")).collect()}
for k, v in valuta_counts.items():
    print("  %-8s %8d" % (k, v))

print("\nStatement type (KONSERN would mean consolidated figures alongside company ones):")
regnskapstype_counts = {r["regnskapstype"]: r["count"]
                        for r in curated.groupBy("regnskapstype").count()
                        .orderBy(F.desc("count")).collect()}
for k, v in regnskapstype_counts.items():
    print("  %-12s %8d" % (k, v))

Successful filings in mirror : 444645
Rows in curated table        : 444645  OK
Distinct organisasjonsnummer : 444645  OK

Operating margin status over all 444645 filings:
  computed             293155   65.93%
  revenue_missing       91026   20.47%
  revenue_zero          59206   13.32%
  revenue_negative        875    0.20%
  income_missing          383    0.09%

Comparability flags:
  avviklingsregnskap   true in      317    0.07%
  is_full_year         true in   425548   95.71%
  morselskap           true in    57002   12.82%
  smaa_foretak         true in   434467   97.71%

Currency (a ratio is scale-invariant, so this does not distort the margin):
  NOK        443461
  USD           670
  EUR           332
  SEK            97
  DKK            54
  GBP            20
  JPY             2
  EDT             2
  SGD             2
  CHF             2
  CAD             1
  HKD             1
  PLZ             1

Statement type (KONSERN would mean consolidated figures alongside company one

## Distribution of the computed margin

Reported with medians and percentiles rather than a mean. The margin is
unbounded below and heavily skewed, so a mean is dominated by a handful of
microscopic-revenue companies and describes nothing.

The pooled margin below is the economically meaningful aggregate. The gap
between it and the median is the point worth discussing in the report.

In [4]:
computed = curated.filter("operating_margin_status = 'computed'")

qs = computed.approxQuantile("operating_margin_pct",
                             [0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999],
                             0.0)
labels = ["p0.1", "p1", "p5", "p25", "median", "p75", "p95", "p99", "p99.9"]
print("Per-company operating margin (%), computed rows only:")
for lab, q in zip(labels, qs):
    print("  %-8s %14.2f" % (lab, q))

extremes = computed.agg(F.min("operating_margin_pct"), F.max("operating_margin_pct"),
                        F.mean("operating_margin_pct")).collect()[0]
print("  %-8s %14.2f" % ("min", extremes[0]))
print("  %-8s %14.2f" % ("max", extremes[1]))
print("  %-8s %14.2f   <- reported only to show why it is not used" % ("mean", extremes[2]))

pooled = computed.agg(F.sum("operating_income"), F.sum("revenue")).collect()[0]
print("\nPooled margin  SUM(operating_income)/SUM(revenue)*100 = %.2f%%"
      % (100.0 * pooled[0] / pooled[1]))

Per-company operating margin (%), computed rows only:
  p0.1          -50338.89
  p1             -1974.82
  p5              -192.13
  p25               -4.13
  median             6.53
  p75               31.63
  p95               79.27
  p99               97.56
  p99.9            184.22
  min      -2731645700.00
  max          1800100.00
  mean          -11786.56   <- reported only to show why it is not used

Pooled margin  SUM(operating_income)/SUM(revenue)*100 = 5.51%


## Write

`coalesce(1)` into container-local scratch, then copy the single part file onto
the Dropbox-synced mount. At this row count the cost of collapsing to one
partition is negligible, and it gives PowerBI a plain file path with no
Folder/Combine step.

Scratch and deliverable deliberately live on different filesystems. See the
filesystem note at the top.

In [5]:
if os.path.exists(STAGING_DIR):
    shutil.rmtree(STAGING_DIR, ignore_errors=True)

curated.coalesce(1).write.mode("overwrite").parquet(STAGING_DIR)

parts = glob.glob(os.path.join(STAGING_DIR, "part-*.parquet"))
assert len(parts) == 1, "expected exactly one part file, got %d" % len(parts)

if os.path.exists(ANALYTICS_FILE):
    os.remove(ANALYTICS_FILE)

# copyfile rather than move: /tmp and the bind mount are different filesystems,
# and a copy leaves the source intact if the destination write is interrupted.
shutil.copyfile(parts[0], ANALYTICS_FILE)

# Cleanup is on container-local disk, but stays non-fatal either way: a failure
# here leaves scratch behind, it does not invalidate the deliverable.
shutil.rmtree(STAGING_DIR, ignore_errors=True)

print("Wrote %s  (%.1f MB)"
      % (ANALYTICS_FILE, os.path.getsize(ANALYTICS_FILE) / 1024**2))

Wrote /home/jovyan/data/parquet/analytics_company_financials.parquet  (29.6 MB)


## Verification

Re-read from disk rather than trusting the in-memory DataFrame, so a write that
silently dropped rows or columns is caught. The recomputation check is the
important one: it re-derives the margin from the two stored components and
requires agreement to 1e-9 relative, which proves the stored ratio and the
stored numerator and denominator are mutually consistent — the property PowerBI
depends on when it computes pooled margins from the components.

In [6]:
out = spark.read.parquet(ANALYTICS_FILE)

print("Rows on disk: %d  %s" % (out.count(), "OK" if out.count() == n_rows else "MISMATCH"))
assert out.count() == n_rows

# Every null margin must carry a non-'computed' reason, and vice versa.
bad = out.filter(
    (F.col("operating_margin_pct").isNull() & (F.col("operating_margin_status") == "computed"))
    | (F.col("operating_margin_pct").isNotNull() & (F.col("operating_margin_status") != "computed"))
).count()
print("Margin/status contradictions: %d  %s" % (bad, "OK" if bad == 0 else "FAIL"))
assert bad == 0

# Recompute from the stored components and compare.
recomputed = out.filter("operating_margin_status = 'computed'").withColumn(
    "delta",
    F.abs(F.col("operating_margin_pct") - (F.col("operating_income") / F.col("revenue")) * 100)
    / F.greatest(F.abs(F.col("operating_margin_pct")), F.lit(1.0)))
worst = recomputed.agg(F.max("delta")).collect()[0][0]
print("Worst relative recomputation error: %.3e  %s"
      % (worst, "OK" if worst < 1e-9 else "FAIL"))
assert worst < 1e-9

# No computed row may have a non-positive denominator.
assert out.filter("operating_margin_status = 'computed' AND revenue <= 0").count() == 0
print("No computed row has revenue <= 0  OK")

print("\nSchema:")
out.printSchema()

Rows on disk: 444645  OK
Margin/status contradictions: 0  OK
Worst relative recomputation error: 0.000e+00  OK
No computed row has revenue <= 0  OK

Schema:
root
 |-- organisasjonsnummer: string (nullable = true)
 |-- navn: string (nullable = true)
 |-- organisasjonsform_kode: string (nullable = true)
 |-- naeringskode1_kode: string (nullable = true)
 |-- naeringskode1_beskrivelse: string (nullable = true)
 |-- kommunenummer: string (nullable = true)
 |-- kommune: string (nullable = true)
 |-- antall_ansatte: integer (nullable = true)
 |-- stiftelsesdato: string (nullable = true)
 |-- registrert_dato: string (nullable = true)
 |-- konkurs: boolean (nullable = true)
 |-- under_avvikling: boolean (nullable = true)
 |-- regnskapstype: string (nullable = true)
 |-- oppstillingsplan: string (nullable = true)
 |-- valuta: string (nullable = true)
 |-- morselskap: boolean (nullable = true)
 |-- smaa_foretak: boolean (nullable = true)
 |-- avviklingsregnskap: boolean (nullable = true)
 |-- per

## Why a third of filings have no margin

`revenue_missing` is the second-largest status class, so it needs an
explanation rather than a footnote. Two competing hypotheses, both testable
from columns already in the table:

1. **Layout.** The API omits the `driftsinntekter` node for statement layouts
   that do not have one — banks and insurers use a different income statement.
   If so, the missing cases concentrate in particular `oppstillingsplan`
   values and the rest of the statement is present.
2. **Dormant and holding companies.** Income is financial rather than
   operating, so there is no operating revenue to report. If so, the missing
   cases spread across layouts and correlate with legal form.

The second block below discriminates between them: if `operating_costs` and
the balance sheet are present while revenue is not, the node is layout-
specific rather than the filing being empty.

In [7]:
print("Status mix by statement layout:")
(out.groupBy("oppstillingsplan")
    .agg(F.count("*").alias("n"),
         F.sum(F.when(F.col("operating_margin_status") == "revenue_missing", 1)
                .otherwise(0)).alias("missing"),
         F.sum(F.when(F.col("operating_margin_status") == "revenue_zero", 1)
                .otherwise(0)).alias("zero"))
    .withColumn("pct_missing", F.round(100.0 * F.col("missing") / F.col("n"), 2))
    .withColumn("pct_zero", F.round(100.0 * F.col("zero") / F.col("n"), 2))
    .orderBy(F.desc("n")).show(30, truncate=False))

missing = out.filter("operating_margin_status = 'revenue_missing'")
n_missing = missing.count()
print("Within the %d revenue_missing filings, how much else is present:" % n_missing)
for col in ["operating_costs", "operating_income", "sum_eiendeler", "sum_egenkapital"]:
    present = missing.filter(F.col(col).isNotNull()).count()
    print("  %-18s present in %7d  (%5.2f%%)"
          % (col, present, 100.0 * present / n_missing))

print("Legal forms carrying the missing cases:")
missing.groupBy("organisasjonsform_kode").count().orderBy(F.desc("count")).show(10, False)

Status mix by statement layout:
+----------------+------+-------+-----+-----------+--------+
|oppstillingsplan|n     |missing|zero |pct_missing|pct_zero|
+----------------+------+-------+-----+-----------+--------+
|store           |444645|91026  |59206|20.47      |13.32   |
+----------------+------+-------+-----+-----------+--------+

Within the 91026 revenue_missing filings, how much else is present:
  operating_costs    present in   83543  (91.78%)
  operating_income   present in   84024  (92.31%)
  sum_eiendeler      present in   91026  (100.00%)
  sum_egenkapital    present in   89508  (98.33%)
Legal forms carrying the missing cases:
+----------------------+-----+
|organisasjonsform_kode|count|
+----------------------+-----+
|AS                    |87486|
|STI                   |1518 |
|NUF                   |489  |
|BRL                   |425  |
|ESEK                  |246  |
|ANS                   |192  |
|ENK                   |169  |
|DA                    |143  |
|FLI        

## Scalability experiment

The same `build` function run at `local[1]`, `local[2]`, `local[4]`, `local[8]`
and `local[12]` on a 12-core host, ending in a write so shuffle and output cost
are included rather than only the scan.

Method matches `Benchmark_engines.ipynb`: one discarded warm-up run per setting,
then three timed runs, median reported. Nothing is cached during the timed runs,
so each measures a cold build from the Parquet mirror.

`spark.driver.memory` is fixed at 8 GB by `spark-defaults.conf` and is applied
at JVM launch, so it cannot change between settings — the heap is held constant
and thread count is the only variable. That is the controlled comparison we
want. `spark.master`, by contrast, is read when the `SparkContext` is created,
so stopping and recreating the session is enough to change it.

Output goes to `/tmp` inside the container, not the Dropbox-synced `data/`.

**If a session fails to restart**, restart the kernel, re-run the setup cell and
run this cell alone. Recreating a `SparkContext` in a live Python process is
reliable in local mode but is not something Spark guarantees.

In [8]:
THREAD_COUNTS = [1, 2, 4, 8, 12]
REPEATS = 3

# Free the cached build before timing; a cached DataFrame from the earlier
# session would make the first setting look artificially fast.
curated.unpersist()
spark.stop()

scal = {}

for n_threads in THREAD_COUNTS:
    session = (SparkSession.builder
               .appName("group13_scalability_local%d" % n_threads)
               .master("local[%d]" % n_threads)
               .getOrCreate())

    target = os.path.join(SCRATCH_DIR, "local%d" % n_threads)

    def run(session=session, target=target):
        build(session).coalesce(1).write.mode("overwrite").parquet(target)

    try:
        run()                                   # warm-up, discarded
        times = []
        for _ in range(REPEATS):
            t0 = time.perf_counter()
            run()
            times.append(time.perf_counter() - t0)
        scal["local[%d]" % n_threads] = {"times": times, "error": None}
        print("local[%-2d]  median %6.2fs   min %6.2fs   max %6.2fs"
              % (n_threads, statistics.median(times), min(times), max(times)))
    except Exception as exc:                    # noqa: BLE001
        scal["local[%d]" % n_threads] = {"times": [], "error": "%s: %s" % (type(exc).__name__, exc)}
        print("local[%-2d]  DID NOT COMPLETE  %s" % (n_threads, exc))

    session.stop()

if os.path.exists(SCRATCH_DIR):
    shutil.rmtree(SCRATCH_DIR)

local[1 ]  median  10.21s   min   9.63s   max  10.81s
local[2 ]  median   7.37s   min   6.86s   max   7.42s
local[4 ]  median   5.86s   min   5.68s   max   6.05s
local[8 ]  median   5.47s   min   5.46s   max   5.83s
local[12]  median   6.03s   min   5.46s   max   6.19s


## Speedup

Speedup relative to `local[1]`, against the ideal linear line. The gap between
them is the interesting quantity: it is where Amdahl's law, the non-splittable
parts of the job and the single-partition write become visible.

In [9]:
baseline = statistics.median(scal["local[1]"]["times"]) if scal["local[1]"]["times"] else None

print("%-10s %10s %10s %10s %12s" % ("setting", "median s", "speedup", "ideal", "efficiency"))
print("-" * 56)
for n_threads in THREAD_COUNTS:
    key = "local[%d]" % n_threads
    times = scal[key]["times"]
    if not times or baseline is None:
        print("%-10s %10s" % (key, "n/a"))
        continue
    med = statistics.median(times)
    speedup = baseline / med
    print("%-10s %10.2f %10.2fx %9dx %11.0f%%"
          % (key, med, speedup, n_threads, 100.0 * speedup / n_threads))

setting      median s    speedup      ideal   efficiency
--------------------------------------------------------
local[1]        10.21       1.00x         1x         100%
local[2]         7.37       1.39x         2x          69%
local[4]         5.86       1.74x         4x          44%
local[8]         5.47       1.87x         8x          23%
local[12]        6.03       1.69x        12x          14%


## Persist the results

Written next to the benchmark output so the report can cite figures from a file
rather than from a notebook that has since been re-run.

In [10]:
summary = {
    "rows": n_rows,
    "source_filings": n_source,
    "spark_version": SPARK_VERSION,
    "cpu_cores": os.cpu_count(),
    "repeats": REPEATS,
    "thread_counts": THREAD_COUNTS,
    "status_counts": status_counts,
    "flag_counts": flag_counts,
    "valuta_counts": valuta_counts,
    "regnskapstype_counts": regnskapstype_counts,
    "margin_quantiles": dict(zip(labels, qs)),
    "margin_min_pct": float(extremes[0]),
    "margin_max_pct": float(extremes[1]),
    "mean_margin_pct": float(extremes[2]),
    "pooled_margin_pct": 100.0 * pooled[0] / pooled[1],
    "scalability_seconds": scal,
    "output_file": ANALYTICS_FILE,
    "output_bytes": os.path.getsize(ANALYTICS_FILE),
}

path = os.path.join(DATA_DIR, "analytics_build_summary.json")
with open(path, "w") as fh:
    json.dump(summary, fh, indent=2)
print("Wrote", path)

Wrote /home/jovyan/data/analytics_build_summary.json
